# BioListen VN — Advanced Multi-task Model Training [Colab & FPT AI Factory Compatible]

Notebook này thực hiện huấn luyện mô hình **Multi-task nâng cao (Advanced Model)** tích hợp toàn bộ các nguồn dữ liệu phục vụ dự án **BioListen VN** trên hạ tầng GPU hiệu năng cao của FPT AI Factory hoặc Google Colab.

### 💡 Tương thích đa môi trường (Colab & JupyterLab / FPT AI Factory):
- Tự động nhận diện môi trường: Nếu chạy trên Colab sẽ tự động mount Google Drive; nếu chạy trên JupyterLab/FPT AI Factory sẽ sử dụng đường dẫn tương đối `./data` để nạp dữ liệu cục bộ.
- Cấu hình luồng nạp tránh tràn RAM và tối ưu hóa I/O.

## 1. Kết nối Google Drive (Colab) / Cấu hình đường dẫn (JupyterLab)

In [ ]:
import os
import zipfile
import shutil
from tqdm import tqdm

# Tự động phát hiện môi trường chạy
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
    print("✓ Đang chạy trên Google Colab. Kết nối Drive thành công.")
except ImportError:
    IS_COLAB = False
    print("💡 Đang chạy trên JupyterLab / FPT AI Factory (Bỏ qua mount Google Colab).")

# Cấu hình đường dẫn dữ liệu
if IS_COLAB:
    drive_processed_dir = '/content/drive/MyDrive/Datasets/BioListenVN/processed'
    base_models_dir = '/content/drive/MyDrive/Datasets/BioListenVN/models'
    local_fsc22_dir = '/content/fsc22_processed'
    local_esc50_dir = '/content/esc50_processed'
    local_rfcx_tp_dir = '/content/rfcx_processed_tp'
    local_rfcx_fp_dir = '/content/rfcx_processed_fp'
    local_anuraset_dir = '/content/anuraset_processed'
    local_zenodo_dir = '/content/zenodo_processed'
else:
    # FPT AI Factory: Đặt dữ liệu của bạn trong thư mục data/
    DATA_BASE_DIR = './data'
    drive_processed_dir = os.path.join(DATA_BASE_DIR, 'processed')
    base_models_dir = './models'
    local_fsc22_dir = os.path.join(DATA_BASE_DIR, 'fsc22_processed')
    local_esc50_dir = os.path.join(DATA_BASE_DIR, 'esc50_processed')
    local_rfcx_tp_dir = os.path.join(DATA_BASE_DIR, 'rfcx_processed_tp')
    local_rfcx_fp_dir = os.path.join(DATA_BASE_DIR, 'rfcx_processed_fp')
    local_anuraset_dir = os.path.join(DATA_BASE_DIR, 'anuraset_processed')
    local_zenodo_dir = os.path.join(DATA_BASE_DIR, 'zenodo_processed')

def extract_zip(zip_name, dest_dir):
    zip_path = os.path.join(drive_processed_dir, f"{zip_name}.zip")
    if os.path.exists(zip_path):
        # Chỉ giải nén nếu chưa giải nén trước đó
        if not os.path.exists(dest_dir) or len(os.listdir(dest_dir)) == 0:
            print(f"Đang giải nén {zip_name}.zip...")
            os.makedirs(dest_dir, exist_ok=True)
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(dest_dir)
            print(f"-> Đã giải nén ra {dest_dir} (Số file: {len(os.listdir(dest_dir))})")
        else:
            print(f"-> Thư mục {dest_dir} đã có dữ liệu. Bỏ qua giải nén.")
    else:
        print(f"💡 Bỏ qua: Không tìm thấy file lưu trữ tại {zip_path}")

# Thực hiện giải nén
extract_zip('fsc22_processed', local_fsc22_dir)
extract_zip('esc50_processed', local_esc50_dir)
extract_zip('rfcx_tp_processed', local_rfcx_tp_dir)
extract_zip('rfcx_fp_processed', local_rfcx_fp_dir)
extract_zip('anuraset_processed', local_anuraset_dir)
extract_zip('zenodo_processed', local_zenodo_dir)

# Xử lý sao chép metadata CSV trên Colab
metadata_files = [
    'fsc22_processed_metadata.csv',
    'esc50_processed_metadata.csv',
    'rfcx_tp_processed_metadata.csv',
    'rfcx_fp_processed_metadata.csv',
    'anuraset_processed_metadata.csv',
    'zenodo_processed_metadata.csv'
]
if IS_COLAB:
    for csv_name in metadata_files:
        src = os.path.join(drive_processed_dir, csv_name)
        dst = os.path.join('/content', csv_name)
        if os.path.exists(src):
            shutil.copy(src, dst)
            print(f"Đã copy metadata {csv_name} về Colab local.")
else:
    print("✓ Metadata được đọc trực tiếp từ thư mục processed của JupyterLab.")

## 2. Xây dựng Multi-task Dataset & DataLoader

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Cấu hình I/O tối ưu cho phần cứng FPT AI Factory
NUM_WORKERS = 4  # Điều chỉnh tùy số nhân CPU của VM

# Định nghĩa 8 lớp đe dọa thực tế chính
THREAT_CLASSES = [
    'Fire', 'Chainsaw', 'Handsaw', 'Helicopter', 'VehicleEngine', 
    'Axe', 'Gunshot', 'Footsteps'
]
threat_to_idx = {name: idx for idx, name in enumerate(THREAT_CLASSES)}
BACKGROUND_CLASS_IDX = 8

# Ánh xạ các nhãn phụ trợ của ESC-50 sang nhãn mối đe dọa chung
esc50_threat_map = {
    'crackling_fire': 0, # Fire
    'chainsaw': 1,       # Chainsaw
    'hand_saw': 2,       # Handsaw
    'helicopter': 3,     # Helicopter
    'engine': 4,         # VehicleEngine
    'gun_shot': 6,       # Gunshot
    'footsteps': 7       # Footsteps
}

class BioListenMultiTaskDataset(Dataset):
    def __init__(self, samples_list, fsc22_dir, esc50_dir, rfcx_tp_dir, rfcx_fp_dir, anuraset_dir, zenodo_dir):
        self.samples = samples_list
        self.fsc22_dir = fsc22_dir
        self.esc50_dir = esc50_dir
        self.rfcx_tp_dir = rfcx_tp_dir
        self.rfcx_fp_dir = rfcx_fp_dir
        self.anuraset_dir = anuraset_dir
        self.zenodo_dir = zenodo_dir

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        task_type = sample['task_type']
        dataset_name = sample['dataset_name']
        
        if task_type == 'human':
            # 1. Tải tensor đặc trưng
            if dataset_name == 'fsc22':
                pt_path = os.path.join(self.fsc22_dir, sample['processed_pt_filename'])
                tensor = torch.load(pt_path)
                category = sample['Class Name']
                threat_label = threat_to_idx.get(category, BACKGROUND_CLASS_IDX)
            else:  # esc50
                pt_path = os.path.join(self.esc50_dir, sample['processed_pt_filename'])
                tensor = torch.load(pt_path)
                category = sample['category']
                threat_label = esc50_threat_map.get(category, BACKGROUND_CLASS_IDX)
                
            species_label = torch.zeros(24)  # Placeholder
            
        elif task_type == 'species':
            # 2. Tải tensor đặc trưng từ các nguồn loài
            if dataset_name == 'rfcx_tp':
                pt_path = os.path.join(self.rfcx_tp_dir, sample['processed_pt_filename'])
                tensor = torch.load(pt_path)
            elif dataset_name == 'rfcx_fp':
                pt_path = os.path.join(self.rfcx_fp_dir, sample['processed_pt_filename'])
                tensor = torch.load(pt_path)
            elif dataset_name == 'anuraset':
                pt_path = os.path.join(self.anuraset_dir, sample['processed_pt_filename'])
                tensor = torch.load(pt_path)
            else:  # zenodo
                pt_path = os.path.join(self.zenodo_dir, sample['processed_pt_filename'])
                tensor = torch.load(pt_path)
                
            # Cấu hình nhãn loài
            species_label = torch.zeros(24)
            if dataset_name != 'rfcx_fp':  # FP không có nhãn dương tính
                species_id = int(sample['species_id'])
                if 0 <= species_id < 24:
                    species_label[species_id] = 1.0
                    
            threat_label = BACKGROUND_CLASS_IDX  # Placeholder
            
        return tensor, species_label, threat_label, task_type

# Đọc metadata và chuẩn bị danh sách mẫu
all_train_samples = []
all_val_samples = []

# Định nghĩa đường dẫn các file CSV dựa trên môi trường
fsc22_csv = '/content/fsc22_processed_metadata.csv' if IS_COLAB else os.path.join(drive_processed_dir, 'fsc22_processed_metadata.csv')
esc50_csv = '/content/esc50_processed_metadata.csv' if IS_COLAB else os.path.join(drive_processed_dir, 'esc50_processed_metadata.csv')
rfcx_tp_csv = '/content/rfcx_tp_processed_metadata.csv' if IS_COLAB else os.path.join(drive_processed_dir, 'rfcx_tp_processed_metadata.csv')
rfcx_fp_csv = '/content/rfcx_fp_processed_metadata.csv' if IS_COLAB else os.path.join(drive_processed_dir, 'rfcx_fp_processed_metadata.csv')
anura_csv = '/content/anuraset_processed_metadata.csv' if IS_COLAB else os.path.join(drive_processed_dir, 'anuraset_processed_metadata.csv')
zenodo_csv = '/content/zenodo_processed_metadata.csv' if IS_COLAB else os.path.join(drive_processed_dir, 'zenodo_processed_metadata.csv')

# A. Nạp và phân chia FSC22 (80% Train, 20% Val)
if os.path.exists(fsc22_csv):
    df_fsc = pd.read_csv(fsc22_csv)
    df_fsc_train, df_fsc_val = train_test_split(df_fsc, test_size=0.2, random_state=42)
    
    for _, row in df_fsc_train.iterrows():
        s_dict = row.to_dict()
        s_dict['task_type'] = 'human'
        s_dict['dataset_name'] = 'fsc22'
        all_train_samples.append(s_dict)
    for _, row in df_fsc_val.iterrows():
        s_dict = row.to_dict()
        s_dict['task_type'] = 'human'
        s_dict['dataset_name'] = 'fsc22'
        all_val_samples.append(s_dict)
    print(f"FSC22 nạp xong: Train={len(df_fsc_train)} | Val={len(df_fsc_val)}")

# B. Nạp và phân chia ESC-50 phụ trợ (80% Train, 20% Val)
if os.path.exists(esc50_csv):
    df_esc = pd.read_csv(esc50_csv)
    df_esc_train, df_esc_val = train_test_split(df_esc, test_size=0.2, random_state=42)
    
    for _, row in df_esc_train.iterrows():
        s_dict = row.to_dict()
        s_dict['task_type'] = 'human'
        s_dict['dataset_name'] = 'esc50'
        all_train_samples.append(s_dict)
    for _, row in df_esc_val.iterrows():
        s_dict = row.to_dict()
        s_dict['task_type'] = 'human'
        s_dict['dataset_name'] = 'esc50'
        all_val_samples.append(s_dict)
    print(f"ESC-50 nạp xong: Train={len(df_esc_train)} | Val={len(df_esc_val)}")

# C. Nạp và phân chia RFCx TP (80% Train, 20% Val)
if os.path.exists(rfcx_tp_csv):
    df_tp = pd.read_csv(rfcx_tp_csv)
    df_tp_train, df_tp_val = train_test_split(df_tp, test_size=0.2, random_state=42)
    
    for _, row in df_tp_train.iterrows():
        s_dict = row.to_dict()
        s_dict['task_type'] = 'species'
        s_dict['dataset_name'] = 'rfcx_tp'
        all_train_samples.append(s_dict)
    for _, row in df_tp_val.iterrows():
        s_dict = row.to_dict()
        s_dict['task_type'] = 'species'
        s_dict['dataset_name'] = 'rfcx_tp'
        all_val_samples.append(s_dict)
    print(f"RFCx TP nạp xong: Train={len(df_tp_train)} | Val={len(df_tp_val)}")

# D. Nạp và phân chia RFCx FP
if os.path.exists(rfcx_fp_csv):
    df_fp = pd.read_csv(rfcx_fp_csv)
    df_fp_train, df_fp_val = train_test_split(df_fp, test_size=0.2, random_state=42)
    
    for _, row in df_fp_train.iterrows():
        s_dict = row.to_dict()
        s_dict['task_type'] = 'species'
        s_dict['dataset_name'] = 'rfcx_fp'
        all_train_samples.append(s_dict)
    for _, row in df_fp_val.iterrows():
        s_dict = row.to_dict()
        s_dict['task_type'] = 'species'
        s_dict['dataset_name'] = 'rfcx_fp'
        all_val_samples.append(s_dict)
    print(f"RFCx FP nạp xong: Train={len(df_fp_train)} | Val={len(df_fp_val)}")

# E. Tự động nạp thêm Anuraset (Nếu tồn tại file metadata)
if os.path.exists(anura_csv):
    df_anura = pd.read_csv(anura_csv)
    df_anura_train, df_anura_val = train_test_split(df_anura, test_size=0.2, random_state=42)
    for _, row in df_anura_train.iterrows():
        s_dict = row.to_dict()
        s_dict['task_type'] = 'species'
        s_dict['dataset_name'] = 'anuraset'
        all_train_samples.append(s_dict)
    for _, row in df_anura_val.iterrows():
        s_dict = row.to_dict()
        s_dict['task_type'] = 'species'
        s_dict['dataset_name'] = 'anuraset'
        all_val_samples.append(s_dict)
    print(f"Anuraset nạp xong: Train={len(df_anura_train)} | Val={len(df_anura_val)}")

# F. Tự động nạp thêm Zenodo
if os.path.exists(zenodo_csv):
    df_zenodo = pd.read_csv(zenodo_csv)
    df_zen_train, df_zen_val = train_test_split(df_zenodo, test_size=0.2, random_state=42)
    for _, row in df_zen_train.iterrows():
        s_dict = row.to_dict()
        s_dict['task_type'] = 'species'
        s_dict['dataset_name'] = 'zenodo'
        all_train_samples.append(s_dict)
    for _, row in df_zen_val.iterrows():
        s_dict = row.to_dict()
        s_dict['task_type'] = 'species'
        s_dict['dataset_name'] = 'zenodo'
        all_val_samples.append(s_dict)
    print(f"Zenodo nạp xong: Train={len(df_zen_train)} | Val={len(df_zen_val)}")

# Khởi tạo DataLoader
train_dataset = BioListenMultiTaskDataset(
    all_train_samples, local_fsc22_dir, local_esc50_dir, 
    local_rfcx_tp_dir, local_rfcx_fp_dir, local_anuraset_dir, local_zenodo_dir
)
val_dataset = BioListenMultiTaskDataset(
    all_val_samples, local_fsc22_dir, local_esc50_dir, 
    local_rfcx_tp_dir, local_rfcx_fp_dir, local_anuraset_dir, local_zenodo_dir
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=NUM_WORKERS, drop_last=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"DataLoader Khởi tạo xong: Total Train Batches={len(train_loader)} | Total Val Batches={len(val_loader)}")

## 3. Kiến trúc Mô hình Multi-task Nâng cao (Backbone: EfficientNet-V2-S)

In [ ]:
import torchvision.models as models

class BioListenMultiTaskModel(nn.Module):
    def __init__(self, num_species=24, num_threats=9, pretrained=True):
        super().__init__()
        # Tải mô hình EfficientNet-V2-S làm Backbone lớn hơn
        weights = models.EfficientNet_V2_S_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = models.efficientnet_v2_s(weights=weights)
        
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()  # Đưa về Identity
        
        # Nhánh 1: Phân loại loài (species_head) - Output 24
        self.species_head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_species)
        )
        
        # Nhánh 2: Phân loại đe dọa (human_head) - Output 9
        self.human_head = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_threats)
        )

    def forward(self, x):
        features = self.backbone(x)  # Shape: (B, 1280)
        species_logits = self.species_head(features)  # Shape: (B, 24)
        threat_logits = self.human_head(features)  # Shape: (B, 9)
        return species_logits, threat_logits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BioListenMultiTaskModel(pretrained=True).to(device)
print(f"Mô hình nâng cao đã khởi tạo thành công trên thiết bị: {device}")

## 4. Hàm Loss Thích ứng & Thiết lập Mixed Precision (AMP)

In [ ]:
def compute_multitask_loss(species_logits, threat_logits, species_targets, threat_targets, task_types, alpha=1.0, beta=1.0):
    is_species = torch.tensor([t == 'species' for t in task_types], device=species_logits.device)
    is_human = torch.tensor([t == 'human' for t in task_types], device=threat_logits.device)
    
    loss_species = torch.tensor(0.0, device=species_logits.device)
    loss_human = torch.tensor(0.0, device=threat_logits.device)
    
    # 1. BCE Loss cho nhánh loài
    if is_species.any():
        spec_logits_masked = species_logits[is_species]
        spec_targets_masked = species_targets[is_species].float()
        loss_species = nn.BCEWithLogitsLoss()(spec_logits_masked, spec_targets_masked)
        
    # 2. CrossEntropy Loss cho nhánh mối đe dọa con người
    if is_human.any():
        threat_logits_masked = threat_logits[is_human]
        threat_targets_masked = threat_targets[is_human].long()
        loss_human = nn.CrossEntropyLoss()(threat_logits_masked, threat_targets_masked)
        
    total_loss = alpha * loss_species + beta * loss_human
    return total_loss, loss_species, loss_human

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# Sử dụng GradScaler cho chế độ Mixed Precision (AMP)
from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()

print("Optimizer AdamW, Cosine Annealing, và GradScaler (AMP) đã sẵn sàng.")

## 5. Early Stopping & Vòng lặp Huấn luyện tối ưu hiệu năng GPU

In [ ]:
# Xác định thư mục lưu trữ cho phase hiện tại tự động
os.makedirs(base_models_dir, exist_ok=True)

existing_phases = [d for d in os.listdir(base_models_dir) if os.path.isdir(os.path.join(base_models_dir, d)) and d.startswith('phase_')]
phase_nums = []
for p in existing_phases:
    try:
        num = int(p.split('_')[1])
        phase_nums.append(num)
    except ValueError:
        pass

current_phase_num = max(phase_nums) + 1 if phase_nums else 1
current_phase_folder = f"phase_{current_phase_num:02d}"
current_save_dir = os.path.join(base_models_dir, current_phase_folder)
os.makedirs(current_save_dir, exist_ok=True)
print(f"=== THƯ MỤC LƯU TRỮ MODEL CHO LƯỢT CHẠY NÀY: {current_save_dir} ===")

class EarlyStopping:
    def __init__(self, patience=7, delta=1e-4, save_dir=current_save_dir):
        self.patience = patience
        self.delta = delta
        self.save_dir = save_dir
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.save_count = 0

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            print(f"EarlyStopping Counter: {self.counter} / {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0

    def save_checkpoint(self, model):
        os.makedirs(self.save_dir, exist_ok=True)
        self.save_count += 1
        
        latest_path = os.path.join(self.save_dir, 'best_model.pt')
        torch.save(model.state_dict(), latest_path)
        
        indexed_path = os.path.join(self.save_dir, f'best_model_{self.save_count}.pt')
        torch.save(model.state_dict(), indexed_path)
        
        print(f"[CHECKPOINT] Đã lưu mô hình tốt nhất (Lần {self.save_count}):")
        print(f"  - File suy luận: {latest_path}")
        print(f"  - File lịch sử: {indexed_path}")

def train_epoch(model, loader, optimizer, device):
    model.train()
    epoch_loss = 0.0
    spec_losses = 0.0
    human_losses = 0.0
    
    pbar = tqdm(loader, desc='Training Batch', leave=True)
    for step, (tensors, species_labels, threat_labels, task_types) in enumerate(pbar):
        tensors = tensors.to(device)
        species_labels = species_labels.to(device)
        threat_labels = threat_labels.to(device)
        
        optimizer.zero_grad()
        
        # Sử dụng autocast để chạy Mixed Precision
        with autocast():
            spec_logits, threat_logits = model(tensors)
            loss, l_spec, l_human = compute_multitask_loss(
                spec_logits, threat_logits, species_labels, threat_labels, task_types
            )
        
        # Lùi ngược có cân scale
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        spec_losses += l_spec.item()
        human_losses += l_human.item()
        
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'spec_loss': f'{l_spec.item():.4f}',
            'human_loss': f'{l_human.item():.4f}'
        })
        
    num_batches = len(loader)
    return epoch_loss / num_batches, spec_losses / num_batches, human_losses / num_batches

def validate(model, loader, device):
    model.eval()
    val_loss = 0.0
    spec_losses = 0.0
    human_losses = 0.0
    
    pbar = tqdm(loader, desc='Validation Batch', leave=True)
    with torch.no_grad():
        for tensors, species_labels, threat_labels, task_types in pbar:
            tensors = tensors.to(device)
            species_labels = species_labels.to(device)
            threat_labels = threat_labels.to(device)
            
            with autocast():
                spec_logits, threat_logits = model(tensors)
                loss, l_spec, l_human = compute_multitask_loss(
                    spec_logits, threat_logits, species_labels, threat_labels, task_types
                )
            
            val_loss += loss.item()
            spec_losses += l_spec.item()
            human_losses += l_human.item()
            
            pbar.set_postfix({'val_loss': f'{loss.item():.4f}'})
            
    num_batches = len(loader)
    return val_loss / num_batches, spec_losses / num_batches, human_losses / num_batches

EPOCHS = 20  # Nâng lên 20 epochs cho FPT AI Factory
early_stopping = EarlyStopping(patience=5)  # Tăng patience để đảm bảo hội tụ tốt

# Khởi tạo history để vẽ đồ thị
history = {
    'train_loss': [], 'train_spec_loss': [], 'train_human_loss': [],
    'val_loss': [], 'val_spec_loss': [], 'val_human_loss': []
}

print("Bắt đầu huấn luyện mô hình Advanced...")
for epoch in range(EPOCHS):
    train_loss, tr_spec, tr_human = train_epoch(model, train_loader, optimizer, device)
    val_loss, val_spec, val_human = validate(model, val_loader, device)
    
    scheduler.step()
    
    # Lưu lịch sử
    history['train_loss'].append(train_loss)
    history['train_spec_loss'].append(tr_spec)
    history['train_human_loss'].append(tr_human)
    history['val_loss'].append(val_loss)
    history['val_spec_loss'].append(val_spec)
    history['val_human_loss'].append(val_human)
    
    print(f"\n==> Epoch {epoch+1:02d}/{EPOCHS:02d} | "
          f"Train Loss: {train_loss:.4f} (Spec: {tr_spec:.4f}, Human: {tr_human:.4f}) | "
          f"Val Loss: {val_loss:.4f} (Spec: {val_spec:.4f}, Human: {val_human:.4f})")
    
    early_stopping(val_loss, model)
    if early_stopping.early_stop:
        print("Kích hoạt Early Stopping! Huấn luyện dừng sớm.")
        break

## 6. Trực quan hóa Lịch sử Huấn luyện (Results Visualizer)

In [ ]:
# Vẽ đồ thị kết quả huấn luyện (Loss Curves)
import matplotlib.pyplot as plt
import seaborn as sns

epochs_range = range(1, len(history['train_loss']) + 1)

plt.figure(figsize=(18, 5))

# 1. Tổng Loss (Total Loss)
plt.subplot(1, 3, 1)
plt.plot(epochs_range, history['train_loss'], label='Train Loss', marker='o', color='royalblue')
plt.plot(epochs_range, history['val_loss'], label='Val Loss', marker='s', color='orange')
plt.title('Total Multi-task Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# 2. Species Head Loss (BCE)
plt.subplot(1, 3, 2)
plt.plot(epochs_range, history['train_spec_loss'], label='Train Species Loss', marker='o', color='forestgreen')
plt.plot(epochs_range, history['val_spec_loss'], label='Val Species Loss', marker='s', color='limegreen')
plt.title('Species Head Loss (BCE)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# 3. Human Head Loss (CrossEntropy)
plt.subplot(1, 3, 3)
plt.plot(epochs_range, history['train_human_loss'], label='Train Human Loss', marker='o', color='crimson')
plt.plot(epochs_range, history['val_human_loss'], label='Val Human Loss', marker='s', color='hotpink')
plt.title('Human Head Loss (CrossEntropy)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 7. Ước lượng Độ tin cậy bằng MC-Dropout

In [ ]:
def predict_mc_dropout(model, input_tensor, num_passes=15):
    model.train()
    input_tensor = input_tensor.to(device).unsqueeze(0)
    species_probs_list = []
    threat_probs_list = []
    
    with torch.no_grad():
        for _ in range(num_passes):
            spec_logits, threat_logits = model(input_tensor)
            spec_probs = torch.sigmoid(spec_logits).cpu().numpy()[0]
            threat_probs = torch.softmax(threat_logits, dim=1).cpu().numpy()[0]
            
            species_probs_list.append(spec_probs)
            threat_probs_list.append(threat_probs)
            
    species_probs_arr = np.array(species_probs_list)
    threat_probs_arr = np.array(threat_probs_list)
    
    spec_mean = np.mean(species_probs_arr, axis=0)
    spec_std = np.std(species_probs_arr, axis=0)
    
    threat_mean = np.mean(threat_probs_arr, axis=0)
    threat_std = np.std(threat_probs_arr, axis=0)
    
    return spec_mean, spec_std, threat_mean, threat_std

test_dataset = val_dataset
sample_tensor, spec_lbl, threat_lbl, task_type = test_dataset[0]
s_mean, s_std, t_mean, t_std = predict_mc_dropout(model, sample_tensor)

print(f"--- KẾT QUẢ MC-DROPOUT TRÊN MẪU THỬ (Task: {task_type}) ---")
if task_type == 'species':
    best_idx = np.argmax(s_mean)
    print(f"Xác định loài dự đoán tốt nhất: s{best_idx} | Mean Prob: {s_mean[best_idx]:.4f} | Uncertainty (STD): {s_std[best_idx]:.4f}")
else:
    best_idx = np.argmax(t_mean)
    label_name = THREAT_CLASSES[best_idx] if best_idx < len(THREAT_CLASSES) else 'background_normal'
    print(f"Dự đoán đe dọa tốt nhất: {label_name.upper()} | Mean Prob: {t_mean[best_idx]:.4f} | Uncertainty (STD): {t_std[best_idx]:.4f}")

## 8. Giải thích Mô hình bằng Grad-CAM

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_backward_hook(self.save_gradient)
        
    def save_activation(self, module, input, output):
        self.activations = output
        
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]
        
    def generate_heatmap(self, input_tensor, task_type='human', target_class=0):
        self.model.eval()
        input_tensor = input_tensor.to(device).unsqueeze(0)
        
        spec_logits, threat_logits = self.model(input_tensor)
        
        if task_type == 'species':
            score = spec_logits[0, target_class]
        else:
            score = threat_logits[0, target_class]
            
        self.model.zero_grad()
        score.backward()
        
        gradients = self.gradients.cpu().data.numpy()[0]
        activations = self.activations.cpu().data.numpy()[0]
        
        weights = np.mean(gradients, axis=(1, 2))
        
        cam = np.zeros(activations.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * activations[i]
            
        cam = np.maximum(cam, 0)
        
        if cam.max() - cam.min() > 1e-9:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        else:
            cam = np.zeros_like(cam)
            
        return cam

target_conv_layer = model.backbone.features[-1]
grad_cam = GradCAM(model, target_conv_layer)

sample_tensor, spec_lbl, threat_lbl, task_type = test_dataset[0]
target_idx = np.argmax(spec_lbl.numpy()) if task_type == 'species' else threat_lbl

heatmap = grad_cam.generate_heatmap(sample_tensor, task_type=task_type, target_class=target_idx)

import cv2
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(sample_tensor[0].numpy(), cmap='magma', origin='lower')
plt.title("Spectrogram Gốc")
plt.grid(False)

plt.subplot(1, 2, 2)
heatmap_resized = cv2.resize(heatmap, (224, 224))
plt.imshow(sample_tensor[0].numpy(), cmap='magma', origin='lower')
plt.imshow(heatmap_resized, cmap='jet', alpha=0.4, origin='lower')
plt.title(f"Grad-CAM Heatmap Overlay (Task: {task_type.upper()})")
plt.grid(False)
plt.show()